# Etapa 1.1 – Conversão e Sanitização Individual de Planilhas B3 (.xlsx → .parquet & .csv)
**Autor:** Pedro Reis  
**Orientador:** Prof. Dr. [Nome do Orientador]  
**Instituição:** Universidade Federal de Goiás (UFG)  
**Contexto:** Trabalho de Conclusão de Curso (TCC) em Finanças Quantitativas  
**Data:** Junho de 2026  

---

## Objetivo *(Regra 1 — Tell a story for an audience)*

Este notebook executa o **primeiro passo** do pipeline de preparação de dados do TCC. A história que ele conta é a seguinte:

1. **Problema:** as planilhas brutas do Economatica (496 arquivos `.xlsx`) estão em formato proprietário, com metadados nas primeiras linhas e sem padronização de tipos — inviabilizando análises reproduzíveis e performáticas.
2. **Solução:** para cada planilha, extraímos o *Ticker* do nome da planilha ativa, sanitizamos colunas e tipos, removemos linhas sem dados de mercado e persistimos o resultado em **Parquet** (leitura ~80× mais rápida que Excel) e **CSV** (interoperabilidade).
3. **Resultado esperado:** um par de arquivos `.parquet`/`.csv` limpos por **ticker válido** (um ativo pode falhar na validação de colunas e ser descartado, de modo que o número de saídas é menor ou igual ao de planilhas lidas), todos ordenados cronologicamente e livres de *look-ahead bias*, prontos para a etapa de consolidação (`02_01_consolidando_dados.ipynb`). O total exato de tickers gerados é reportado no relatório final desta execução.

> **Público-alvo:** banca avaliadora do TCC e o "eu do futuro" que precisará reexecutar ou auditar este pipeline.

---

## Referencial Metodológico — *Ten Simple Rules for Jupyter Notebooks*

Este notebook foi desenvolvido seguindo as **Ten Simple Rules for Writing and Sharing Computational Analyses in Jupyter Notebooks** (Rule et al., *PLOS Computational Biology*, 2019; [doi:10.1371/journal.pcbi.1007007](https://doi.org/10.1371/journal.pcbi.1007007)).

| Fase | Regra | Aplicação neste notebook |
| :--- | :---- | :----------------------- |
| **I — Organizar e documentar** | 1. *Tell a story for an audience* | Narrativa estruturada em Início → Meio → Fim (seção Objetivo acima) |
| | 2. *Document the process, not just the results* | Decisões de engenharia documentadas em cada markdown (ex.: por que `skiprows=3`, por que `dropna(how='all')`) |
| | 3. *Use cell divisions to make steps clear* | Uma célula = um passo lógico; cabeçalhos descritivos acima de cada bloco de código |
| **II — Qualidade do código** | 4. *Modularize code* | Funções `extrair_ticker_do_nome_da_planilha()` e `processar_e_salvar_dados()` com responsabilidade única, *type hints* e *docstrings* |
| | 5. *Record dependencies* | Versões de Python, Pandas, NumPy e openpyxl impressas na primeira célula |
| | 6. *Use version control* | Notebook versionado via Git no repositório do TCC |
| | 7. *Build a pipeline* | Constantes parametrizadas no topo (`PASTA_LEITURA`, `PASTA_TRATADOS`); execução ponta-a-ponta com `Kernel → Restart & Run All` |
| **III — Compartilhar** | 8. *Share and explain your data* | Origem dos dados (Economatica), estrutura das planilhas e dicionário de colunas documentados abaixo |
| | 9. *Design to be read, run, and explored* | Paralelismo real configurável via `ProcessPoolExecutor` (`max_workers`); relatório de sucessos/falhas ao final |
| | 10. *Advocate for open research* | Código disponível no repositório público; dados proprietários não redistribuídos, mas origem declarada |


In [1]:
import sys
import os
import logging
import multiprocessing as mp
from pathlib import Path
import pandas as pd
import numpy as np
import openpyxl

# Exibição das versões das dependências para fins de auditabilidade
print(f"Versão do Python:   {sys.version}")
print(f"Versão do Pandas:   {pd.__version__}")
print(f"Versão do NumPy:    {np.__version__}")
print(f"Versão do openpyxl: {openpyxl.__version__}")
print(f"Núcleos lógicos disponíveis (os.cpu_count): {os.cpu_count()}")
print(f"Métodos de início de processo suportados:   {mp.get_all_start_methods()}")

Versão do Python:   3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Versão do Pandas:   3.0.3
Versão do NumPy:    2.4.6
Versão do openpyxl: 3.1.5
Núcleos lógicos disponíveis (os.cpu_count): 12
Métodos de início de processo suportados:   ['spawn']


## Configuração do Ambiente e Diretórios *(Regra 7 — Build a pipeline)*

Todas as constantes de caminho são definidas **uma única vez** no topo do notebook, seguindo o princípio de parametrização centralizada. A segregação física dos diretórios (`dados_economatica` → entrada; `dados_economatica_tratados` → saída) blinda o pipeline contra acoplamento e garante que os dados brutos nunca sejam sobrescritos.

*(Regra 2 — Document the process):* o caminho raiz é resolvido via `Path.cwd().parent.parent` porque este notebook reside em `src/01_Conversao_Parquet/`.

In [2]:
# Configuração do logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] - %(message)s"
)

# Obtenção do caminho raiz do projeto
# Considerando que este script está em src/01_Conversao_Parquet/
PASTA_PROJETO = Path.cwd().parent.parent

PASTA_LEITURA = PASTA_PROJETO / "data" / "dados_economatica"
PASTA_TRATADOS = PASTA_PROJETO / "data" / "dados_economatica_tratados"

# Criação automatizada do diretório de destino se não existir
PASTA_TRATADOS.mkdir(parents=True, exist_ok=True)

print(f"[OK] Pasta de Leitura:   {PASTA_LEITURA.resolve()}")
print(f"[OK] Pasta de Tratados:  {PASTA_TRATADOS.resolve()}")

[OK] Pasta de Leitura:   C:\VSCodeWorkspace\1_TCC_Final\data\dados_economatica
[OK] Pasta de Tratados:  C:\VSCodeWorkspace\1_TCC_Final\data\dados_economatica_tratados


## Funções de Ingestão e Sanitização Individual *(Regra 4 — Modularize code)*

A lógica de ETL foi decomposta em duas funções com responsabilidade única:

| Função | Responsabilidade |
| :----- | :--------------- |
| `extrair_ticker_do_nome_da_planilha()` | Abre o `.xlsx` em modo `read_only` (economia de memória) e extrai o nome do ativo a partir do **nome da planilha ativa** (a aba) |
| `processar_e_salvar_dados()` | Lê os dados a partir da linha 4, valida colunas, converte tipos, remove linhas sem dados de mercado e persiste em Parquet/CSV |

*(Regra 2 — Document the process):* decisões de engenharia relevantes:
- **`skiprows=3`:** as 3 primeiras linhas das planilhas Economatica contêm metadados (nome do ativo, classe e fonte), não dados tabulares.
- **`na_values=['-']`:** o Economatica usa o caractere `-` para representar valores ausentes.
- **`dropna(subset=colunas_esperadas[1:], how='all')`:** remove linhas que possuem data preenchida mas nenhum dado de mercado — registros-fantasma comuns em séries do Economatica.
- **`sort_values(by='Data')`:** ordenação cronológica explícita e rígida para evitar *look-ahead bias* em cálculos sequenciais posteriores.

> **Nota sobre serialização (Regra 4):** estas funções são definidas em nível de módulo (escopo global do notebook) justamente para que possam ser serializadas e enviadas aos processos-filho pelo `ProcessPoolExecutor` na próxima seção.

*(Regra 8 — Share and explain your data):* dicionário das colunas esperadas:

| Coluna | Descrição | Tipo |
| :----- | :-------- | :--- |
| `Data` | Data do pregão | `datetime64` |
| `Q Negs` | Quantidade de negócios no dia | `float64` |
| `Q Títs` | Quantidade de títulos negociados | `float64` |
| `Volume$` | Volume financeiro em R$ | `float64` |
| `Fechamento` | Preço de fechamento ajustado por dividendos | `float64` |
| `Abertura` | Preço de abertura | `float64` |
| `Mínimo` | Preço mínimo do dia | `float64` |
| `Máximo` | Preço máximo do dia | `float64` |
| `Médio` | Preço médio do dia | `float64` |

In [3]:
def extrair_ticker_do_nome_da_planilha(caminho_arquivo: Path) -> str:
    """
    Abre uma planilha Excel em modo 'read_only' para extrair o ticker a partir do nome da planilha ativa (aba).

    Args:
        caminho_arquivo (Path): Caminho para a planilha .xlsx.

    Returns:
        str: Ticker do ativo limpo (ex: 'PETR4').
    """
    # Abre o arquivo com openpyxl de forma eficiente para ler o nome da aba
    livro = openpyxl.load_workbook(caminho_arquivo, read_only=True)
    planilha = livro.active

    nome_aba = planilha.title
    livro.close()

    if not nome_aba or not isinstance(nome_aba, str):
        raise ValueError(f"O nome da planilha ativa está vazio ou não contém texto no arquivo: {caminho_arquivo.name}")

    # Limpeza: pega o primeiro token (ex: 'PETR4' de 'PETR4 PN' ou 'VALE3' de 'VALE3 ON')
    ticker = nome_aba.split()[0].strip().upper()

    # Remove caracteres que não sejam alfanuméricos
    ticker = "".join(char for char in ticker if char.isalnum())

    return ticker

def processar_e_salvar_dados(caminho_arquivo: Path, pasta_destino: Path) -> str:
    """
    Lê os dados da planilha Economatica a partir da linha 4 (cabeçalho),
    limpa e salva em Parquet e CSV individuais na pasta_destino.

    Args:
        caminho_arquivo (Path): Caminho do arquivo Excel de origem.
        pasta_destino (Path): Pasta de destino para os arquivos Parquet/CSV.

    Returns:
        str: O Ticker extraído do ativo.
    """
    ticker = extrair_ticker_do_nome_da_planilha(caminho_arquivo)

    # Leitura do Excel a partir da linha 4 (0-indexed index 3)
    # O pandas pula as 3 primeiras linhas ('skiprows=3') e lê o cabeçalho completo.
    # Converte '-' em NaN de forma explícita
    df = pd.read_excel(
        caminho_arquivo,
        skiprows=3,
        header=0,
        na_values=['-'],
        keep_default_na=True
    )

    # Tratamento e validação de nomes das colunas
    colunas_esperadas = ['Data', 'Q Negs', 'Q Títs', 'Volume$', 'Fechamento', 'Abertura', 'Mínimo', 'Máximo', 'Médio']

    # Ajusta espaçamentos nas colunas do Excel lido
    df.columns = [c.strip() for c in df.columns]

    # Verifica se as colunas essenciais estão presentes
    for col in colunas_esperadas:
        if col not in df.columns:
            raise KeyError(f"Coluna esperada '{col}' não foi encontrada no arquivo: {caminho_arquivo.name}")

    # Seleciona apenas as colunas esperadas
    df = df[colunas_esperadas].copy()

    # Coerção explícita de tipos de dados
    df['Data'] = pd.to_datetime(df['Data'], errors='coerce')

    # Remove registros com Data nula (linhas vazias no Excel)
    df = df.dropna(subset=['Data'])

    for col in colunas_esperadas[1:]:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Remove registros onde todas as colunas de dados (exceto a Data) estão nulas
    df = df.dropna(subset=colunas_esperadas[1:], how='all')

    # ORDENAÇÃO TEMPORAL EXPLÍCITA E RÍGIDA
    # Passo fundamental para evitar look-ahead bias em cálculos sequenciais ou filtros
    df = df.sort_values(by="Data").reset_index(drop=True)

    # Persistência em múltiplos formatos
    pasta_ticker = pasta_destino / ticker
    pasta_ticker.mkdir(parents=True, exist_ok=True)
    caminho_parquet = pasta_ticker / f"{ticker}.parquet"
    caminho_csv = pasta_ticker / f"{ticker}.csv"

    df.to_parquet(caminho_parquet, index=False)
    df.to_csv(caminho_csv, index=False, sep=";", encoding="utf-8")

    return ticker

## Loop Principal de Processamento *(Regra 9 — Design to be read, run, and explored)*

Varredura de arquivos Excel na pasta de origem e execução em paralelo usando `ThreadPoolExecutor` para otimizar I/O e a carga de CPU associada ao *parsing* do formato XML do Excel.

A abordagem com ThreadPoolExecutor obteve desempenho satisfatório com alto uso de CPU sem incorrer nos entraves de compatibilidade e serialização (*pickling*) observados com ProcessPoolExecutor no Windows.

In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# Varredura de arquivos Excel na pasta de origem (ignorando arquivos temporários lockfiles)
arquivos_excel = [
    p for p in PASTA_LEITURA.glob("*.xlsx")
    if not p.name.startswith("~$")
]

print(f"Total de planilhas identificadas para conversão: {len(arquivos_excel)}")

sucessos = 0
falhas = 0
lista_tickers = []

def processar_wrapper(arq):
    try:
        ticker = processar_e_salvar_dados(arq, PASTA_TRATADOS)
        return True, ticker, None
    except Exception as e:
        return False, arq.name, str(e)

# Execução em paralelo usando ThreadPoolExecutor
max_workers = 8  # Ajuste conforme o número de threads suportadas pela CPU
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(processar_wrapper, arq): arq for arq in arquivos_excel}
    
    for fut in as_completed(futures):
        sucesso, identificador, erro = fut.result()
        if sucesso:
            lista_tickers.append(identificador)
            sucessos += 1
        else:
            logging.error(f"Falha ao processar arquivo {identificador}: {erro}")
            falhas += 1

print("=" * 60)
print(f"EXECUÇÃO DO ETL INDIVIDUAL CONCLUÍDA:")
print(f"  - Sucessos na conversão: {sucessos}")
print(f"  - Falhas na conversão:   {falhas}")
print(f"  - Tickers gerados:       {len(set(lista_tickers))}")
print("=" * 60)

Total de planilhas identificadas para conversão: 496
Contexto de multiprocessing disponível: spawn
Método 'fork' indisponível neste SO (nt). Para evitar falhas de serialização (spawn),
executando com ThreadPoolExecutor usando 8 workers...


EXECUÇÃO DO ETL INDIVIDUAL CONCLUÍDA:
  - Workers:               8
  - Sucessos na conversão: 496
  - Falhas na conversão:   0
  - Tickers gerados:       496
